In [ ]:
#kalyani_23070521036
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# =====================================================================
# 1. DATA PREPROCESSING AND NORMALIZATION
# =====================================================================
def load_and_preprocess_data(filepath_or_url):
    """
    Loads the USA Housing dataset, drops unnecessary columns, splits into
    features/target, normalizes the data, and returns PyTorch tensors.
    """
    # Load dataset
    df = pd.read_csv("/content/housing.csv")

    # Preprocessing: Drop the textual 'Address' column as it cannot be directly fed to FNN
    if 'Address' in df.columns:
        df = df.drop(columns=['Address'])

    # Split features (X) and target (y)
    X = df.drop(columns=['Price']).values
    y = df['Price'].values.reshape(-1, 1)

    # Train-test split (80% train, 20% test)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Normalization using StandardScaler (Zero mean, Unit variance)
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    X_train_scaled = scaler_X.fit_transform(X_train)
    X_test_scaled = scaler_X.transform(X_test)

    y_train_scaled = scaler_y.fit_transform(y_train)
    y_test_scaled = scaler_y.transform(y_test)

    # Convert numpy arrays to PyTorch Tensors
    X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train_scaled, dtype=torch.float32)
    X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
    y_test_tensor = torch.tensor(y_test_scaled, dtype=torch.float32)

    return X_train_tensor, X_test_tensor, y_train_tensor, y_test_tensor, scaler_y

# =====================================================================
# 2. DESIGNING DEEP FEED FORWARD ARCHITECTURES
# =====================================================================
class DynamicFNN(nn.Module):
    """
    A configurable Feedforward Neural Network that adapts to different layer structures.
    """
    def __init__(self, input_dim, hidden_layers):
        super(DynamicFNN, self).__init__()
        layers = []
        in_dim = input_dim

        # Build hidden structural layers dynamically
        for hidden_dim in hidden_layers:
            layers.append(nn.Linear(in_dim, hidden_dim))
            layers.append(nn.ReLU())
            in_dim = hidden_dim

        # Output layer for regression (Predicting single continuous variable: Price)
        layers.append(nn.Linear(in_dim, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

# =====================================================================
# 3. TRAINING & EVALUATION METRICS LOOP
# =====================================================================
def compute_mle_loss(y_true, y_pred):
    """
    Computes a negative log-likelihood variant assuming errors are normally distributed.
    Maximizing Likelihood Estimator (MLE) is mathematically equivalent to minimizing MSE
    plus an variance log parameter under Gaussian assumption.
    """
    variance = torch.mean((y_true - y_pred) ** 2)
    # Form of Gaussian log-likelihood constant omission
    mle_loss = 0.5 * torch.log(variance) + 0.5 * (torch.mean((y_true - y_pred) ** 2) / variance)
    return mle_loss.item()

def train_and_evaluate(X_train, X_test, y_train, y_test, hidden_configs, scaler_y, epochs=150, lr=0.01):
    input_dim = X_train.shape[1]
    results = {}

    for name, layers in hidden_configs.items():
        print(f"\n--- Training Architecture: {name} {layers} ---")
        model = DynamicFNN(input_dim, layers)
        criterion = nn.MSELoss()
        optimizer = optim.Adam(model.parameters(), lr=lr)

        # Training loop
        model.train()
        for epoch in range(epochs):
            optimizer.zero_grad()
            predictions = model(X_train)
            loss = criterion(predictions, y_train)
            loss.backward()
            optimizer.step()

        # Evaluation mode
        model.eval()
        with torch.no_grad():
            # Get predictions on normalized test data
            test_preds_norm = model(X_test)

            # Inverse transform back to true dollar prices for authentic evaluation
            y_pred_actual = scaler_y.inverse_transform(test_preds_norm.numpy())
            y_true_actual = scaler_y.inverse_transform(y_test.numpy())

            # Evaluate explicit evaluation metrics
            mse = np.mean((y_true_actual - y_pred_actual) ** 2)
            rmse = np.sqrt(mse)
            mle = compute_mle_loss(torch.tensor(y_true_actual), torch.tensor(y_pred_actual))

            results[name] = {"MSE": mse, "RMSE": rmse, "MLE (Log Metric)": mle}
            print(f"Results for {name}: MSE={mse:.4f}, RMSE={rmse:.4f}, MLE Loss={mle:.4f}")

    return results

# =====================================================================
# 4. EXECUTION ENTRANCE
# =====================================================================
if __name__ == "__main__":
    # Substitute path with your local dataset trajectory if not running inside Kaggle environment
    # kaggle_dataset_url = "../input/housingcsv/housing.csv"

    # Synthetic generator snippet mimicking the structure for isolated validation run
    print("Generating simulated runtime copy matching Kaggle's USA Housing structure...")
    np.random.seed(42)
    mock_data = pd.DataFrame({
        'Avg. Area Income': np.random.uniform(50000, 90000, 1000),
        'Avg. Area House Age': np.random.uniform(3, 8, 1000),
        'Avg. Area Number of Rooms': np.random.uniform(5, 9, 1000),
        'Avg. Area Number of Bedrooms': np.random.uniform(2, 6, 1000),
        'Area Population': np.random.uniform(20000, 60000, 1000),
        'Price': np.random.uniform(500000, 2000000, 1000),
        'Address': ["Mock Address"] * 1000
    })
    mock_data.to_csv("housing.csv", index=False)

    # 1. Process dataset
    X_train, X_test, y_train, y_test, scaler_y = load_and_preprocess_data("housing.csv")

    # 2. Explore multiple structural depths (Layers expansion optimization)
    architectures = {
        "Shallow Network (1 Hidden Layer)": [32],
        "Medium Network (2 Hidden Layers)": [64, 32],
        "Deep Network (4 Hidden Layers)": [128, 64, 32, 16]
    }

    # 3. Model execution
    summary_metrics = train_and_evaluate(X_train, X_test, y_train, y_test, architectures, scaler_y)

    # 4. Final summary output
    print("\n================== ARCHITECTURE COMPARISON SUMMARY ==================")
    df_summary = pd.DataFrame(summary_metrics).T
    print(df_summary)


Generating simulated runtime copy matching Kaggle's USA Housing structure...

--- Training Architecture: Shallow Network (1 Hidden Layer) [32] ---
Results for Shallow Network (1 Hidden Layer): MSE=232416788480.0000, RMSE=482096.2500, MLE Loss=13.5859

--- Training Architecture: Medium Network (2 Hidden Layers) [64, 32] ---
Results for Medium Network (2 Hidden Layers): MSE=265347678208.0000, RMSE=515119.0938, MLE Loss=13.6522

--- Training Architecture: Deep Network (4 Hidden Layers) [128, 64, 32, 16] ---
Results for Deep Network (4 Hidden Layers): MSE=333103235072.0000, RMSE=577150.9375, MLE Loss=13.7659

================== ARCHITECTURE COMPARISON SUMMARY ==================
                                           MSE          RMSE  MLE (Log Metric)
Shallow Network (1 Hidden Layer)  2.324168e+11  482096.25000         13.585899
Medium Network (2 Hidden Layers)  2.653477e+11  515119.09375         13.652153
Deep Network (4 Hidden Layers)    3.331032e+11  577150.93750         13.765859
